# Module 4: Fine-tuning FinBERT for financial sentiment

This trains the sentiment model that feeds the trading agent. It fine-tunes FinBERT on the Financial
PhraseBank, evaluates it, checks how it generalizes to informal tweets, and shows the directional
sentiment score the rest of the system uses.

**Runtime:** set a GPU first (Runtime > Change runtime type > T4 GPU). Training takes a few minutes.

The heavy lifting lives in `src/sentinel/models/sentiment/`; this notebook just calls it.

In [ ]:
!pip -q install -U transformers datasets evaluate accelerate scikit-learn

import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/rdhanase/sentinel-agentic-trading.git"
REPO_DIR = "sentinel-agentic-trading"

if IN_COLAB:
    if not pathlib.Path(REPO_DIR).exists():
        subprocess.run(["git", "clone", "--branch", "dev", REPO_URL], check=True)
    os.chdir(REPO_DIR)

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import torch
print("GPU available:", torch.cuda.is_available())

In [ ]:
from sentinel.models.sentiment.finetune import finetune
from sentinel.models.sentiment.data import LABELS, load_twitter_ood
from sentinel.models.sentiment.infer import SentimentScorer

## Fine-tune

Four epochs on the 66%-agreement Financial PhraseBank. The function returns the trainer, the tokenized
splits, and the held-out test metrics.

In [ ]:
trainer, enc, metrics = finetune(epochs=4, agreement="sentences_66agree")
print({k: round(v, 4) for k, v in metrics.items() if isinstance(v, float)})

## Evaluate on the held-out test set

Accuracy alone hides the neutral-heavy skew, so look at the per-class report and the confusion matrix.

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

pred = trainer.predict(enc["test"])
y_pred = pred.predictions.argmax(-1)
y_true = pred.label_ids
print(classification_report(y_true, y_pred, target_names=LABELS))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=LABELS, cmap="Blues")
plt.title("FinBERT sentiment: test confusion matrix"); plt.show()

## Out-of-distribution check

The model was trained on formal analyst-style sentences. Tweets are messier, so this is a fair test of
whether it generalizes.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

ood = load_twitter_ood()
scorer = SentimentScorer("models_store/finbert-sentiment")
probs = scorer.predict_proba(list(ood["text"]))
yp = probs.argmax(-1)
print("OOD accuracy:", round(accuracy_score(ood["labels"], yp), 4),
      "| macro-F1:", round(f1_score(ood["labels"], yp, average="macro"), 4))

## The signal the agent consumes

Downstream, the risk/decision agent does not want three probabilities, it wants one number. `score`
returns P(positive) - P(negative), in [-1, 1].

In [ ]:
headlines = [
    "Company beats earnings expectations and raises full-year guidance",
    "Regulators open a fraud probe into the firm as shares tumble",
    "Stock little changed as investors await the Fed decision",
]
for h, s in zip(headlines, scorer.score(headlines)):
    print(f"{s:+.2f}   {h}")

## Save the model

Keep the fine-tuned weights in Drive so the later notebooks can load them without retraining.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# import shutil
# shutil.copytree('models_store/finbert-sentiment',
#                 '/content/drive/MyDrive/sentinel/finbert-sentiment', dirs_exist_ok=True)

**Next (Module 5):** train the volatility circuit-breaker on the price dataset and compare a neural
classifier against an XGBoost baseline, then wire both models into the agent.